In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]
df_day1 = spark.createDataFrame(
passengers_day1,
columns
)

In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
passengers_day2,
columns
)

In [0]:
df_day1.write \
    .format("delta") \
    .save("/tmp/passengers_delta")

In [0]:
df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:

passenger_df = spark.read.format("delta") \
.load("/tmp/passengers_delta")

display(passenger_df)


passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
passenger_df.count()

5

In [0]:
%sql
DESCRIBE HISTORY passengers_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T10:42:17.000Z,148655192397593,azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2149914595525119),8c7d4af7-d424-4654-8c0f-07f79f9d3525,0617-101018-t7feqyxd-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1857)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/passengers_delta"
)

delta_table.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.format("delta") \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 102") \
.show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta") \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 106") \
.show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
%sql
SELECT *
FROM passengers_delta VERSION AS OF 0
WHERE passenger_id = 104

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+



In [0]:
spark.read.format("delta") \
.load("/tmp/passengers_delta") \
.filter("passenger_id = 104") \
.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
%sql
OPTIMIZE passengers_delta;

path,metrics
abfss://unity-catalog-storage@dbstoragepmjizljswznyy.dfs.core.windows.net/7405619327475617/__unitystorage/catalogs/e2071e0c-5582-42b9-98c5-8e7eb9e351e2/tables/73771ad6-e5b0-4f39-be09-51830a8c914b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781693797646, 1781693798159, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE passengers_delta
ZORDER BY (city);

path,metrics
abfss://unity-catalog-storage@dbstoragepmjizljswznyy.dfs.core.windows.net/7405619327475617/__unitystorage/catalogs/e2071e0c-5582-42b9-98c5-8e7eb9e351e2/tables/73771ad6-e5b0-4f39-be09-51830a8c914b,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1857), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781693842262, 1781693842715, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
DELETE FROM passengers_delta
WHERE passenger_id = 105;

num_affected_rows
1


In [0]:
%sql
DESCRIBE HISTORY passengers_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-06-17T10:57:46.000Z,148655192397593,azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2149914595525119),ad7ce553-95dc-497b-bf34-b8c42044ddb3,0617-101018-t7feqyxd-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1857, p25FileSize -> 1809, numDeletionVectorsRemoved -> 1, minFileSize -> 1809, numAddedFiles -> 1, maxFileSize -> 1809, p75FileSize -> 1809, p50FileSize -> 1809, numAddedBytes -> 1809)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-17T10:57:44.000Z,148655192397593,azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#15579L = 105)""])",null,List(2149914595525119),ad7ce553-95dc-497b-bf34-b8c42044ddb3,0617-101018-t7feqyxd-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1720, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1289, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 429)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T10:42:17.000Z,148655192397593,azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2149914595525119),8c7d4af7-d424-4654-8c0f-07f79f9d3525,0617-101018-t7feqyxd-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1857)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
%sql
VACUUM passengers_delta;

path
abfss://unity-catalog-storage@dbstoragepmjizljswznyy.dfs.core.windows.net/7405619327475617/__unitystorage/catalogs/e2071e0c-5582-42b9-98c5-8e7eb9e351e2/tables/73771ad6-e5b0-4f39-be09-51830a8c914b
